# Hurricane Melissa Forest Greening: Land-Cover Analysis

This notebook identifies the 2013 land-cover classes associated with post-Hurricane Melissa NDVI greening in forest-related areas, with a focus on river-flood restoration-benefit pixels. It distinguishes between:

- river-flood restoration-benefit pixels with relative NDVI increase >=10%;
- the stricter forest-equivalent >10% greening raster overlapped with river-flood restoration-benefit pixels; and
- the existing all-Jamaica weighted forest-equivalent greening table.


In [ ]:
from pathlib import Path
import ast

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 7})


In [ ]:
BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"
COMMON = BASE / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "forest_greening_landcover"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDCOVER_PATH = COMMON / "common_incoming_data" / "landcover" / "2013_landcover" / "2013_landuse_LandCover.shp"
ROBYN_DEFS_PATH = BASE / "robyns_libraries" / "Robyn_catchment_analysis.py"
RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
NDVI_BEFORE_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
NDVI_AFTER_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
FOREST_CHANGE_GT10_PATH = PAPER3 / "results" / "threats" / "ndvi" / "draft_processed_images" / "ndvi_forests_relative_substantial_change_gt10_classes_epsg3448.tif"
FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH = PAPER3 / "results" / "threats" / "ndvi" / "draft_processed_images" / "ndvi_forests_increase_by_landuse_class_weighted.csv"

J2USD = 1.0 / 150.0
REL_BASELINE_MIN = 0.20
REL_GREENING_THRESHOLD = 0.10
FIGURE_DPI = 300

input_paths = [
    LANDCOVER_PATH,
    ROBYN_DEFS_PATH,
    RIVER_EAD_MIN_PATH,
    RIVER_EAD_MAX_PATH,
    NDVI_BEFORE_PATH,
    NDVI_AFTER_PATH,
    FOREST_CHANGE_GT10_PATH,
    FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH,
]
for input_path in input_paths:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUT_DIR


In [ ]:
def parse_forest_fraction_definitions(path: Path) -> tuple[set[str], dict[str, dict[str, float]]]:
    # Parse forest-equivalent classes and primary mixed-class fractions from Robyn's shared definitions.
    source_text = path.read_text(encoding="utf-8")
    module = ast.parse(source_text)
    forest_flood_equivalent_classes = None
    mixed_land_use_fractions_primary = None

    for node in module.body:
        if not isinstance(node, ast.Assign):
            continue
        for target in node.targets:
            if not isinstance(target, ast.Name):
                continue
            if target.id == "forest_flood_equivalent_classes" and forest_flood_equivalent_classes is None:
                forest_flood_equivalent_classes = set(ast.literal_eval(node.value))
            if target.id == "mixed_land_use_fractions" and mixed_land_use_fractions_primary is None:
                candidate = ast.literal_eval(node.value)
                has_forest_flood_key = any(
                    isinstance(value, dict) and "forest_flood_equivalent_classes" in value
                    for value in candidate.values()
                )
                if isinstance(candidate, dict) and has_forest_flood_key:
                    mixed_land_use_fractions_primary = candidate

    if forest_flood_equivalent_classes is None or mixed_land_use_fractions_primary is None:
        raise ValueError("Could not parse forest-fraction definitions from Robyn_catchment_analysis.py")
    return forest_flood_equivalent_classes, mixed_land_use_fractions_primary


def forest_fraction_for_class(
    class_name: str,
    forest_flood_equivalent_classes: set[str],
    mixed_land_use_fractions_primary: dict[str, dict[str, float]],
) -> float:
    # Return the forest-equivalent fraction used in the existing forest NDVI analysis.
    if class_name in mixed_land_use_fractions_primary:
        return float(mixed_land_use_fractions_primary[class_name].get("forest_flood_equivalent_classes", 0.0))
    if class_name in forest_flood_equivalent_classes:
        return 1.0
    return 0.0


def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    # Read avoided EAD, convert JMD to USD, and set non-positive pixels to NaN.
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * J2USD

    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")

    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan
    return ead_array, profile


def reproject_continuous_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    # Reproject a continuous raster to the river restoration-benefit grid.
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        np.nan,
        dtype="float32",
    )
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def reproject_categorical_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    # Reproject a categorical raster to the river restoration-benefit grid.
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        -9999,
        dtype="int16",
    )
    with rasterio.open(path) as source_raster:
        source_nodata = source_raster.nodata if source_raster.nodata is not None else -9999
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=-9999,
            resampling=Resampling.nearest,
        )
    return destination


def ead_sum_usd(ead_array: np.ndarray, mask: np.ndarray) -> float:
    # Sum positive avoided EAD values inside a boolean mask.
    return float(np.nansum(np.where(mask & np.isfinite(ead_array), ead_array, 0.0)))


def landcover_group_for_class(class_name: str) -> str:
    # Group detailed 2013 land-cover classes for compact interpretation.
    if "Open dry forest" in class_name:
        return "Open dry forest"
    if class_name == "Plantation: Tree crops, shrub crops, sugar cane, banana" or "Hardwood Plantation" in class_name:
        return "Plantation / tree crops"
    if class_name in {
        "Secondary Forest",
        "Disturbed broadleaved forest (Secondary Forest)",
        "Closed broadleaved forest (Primary Forest)",
    }:
        return "Secondary / broadleaved forest"
    if class_name in {
        "Fields and Secondary Forest",
        "Bamboo and Secondary Forest",
        "Bamboo and Fields",
        "Fields  and Bamboo",
        "Fields or Secondary Forest/Pine Plantation",
    }:
        return "Mixed fields, bamboo and secondary forest"
    if class_name.startswith("Fields:"):
        return "Open / agricultural fields"
    if class_name in {"Bauxite Extraction", "Quarry"}:
        return "Bauxite extraction / quarry"
    if class_name == "Bamboo":
        return "Bamboo"
    return "Other"


def pct(numerator: float, denominator: float) -> float:
    # Return percentage, preserving NaN when the denominator is zero.
    return float(numerator / denominator * 100.0) if denominator else np.nan


In [ ]:
forest_flood_equivalent_classes, mixed_land_use_fractions_primary = parse_forest_fraction_definitions(ROBYN_DEFS_PATH)

river_ead_min_usd, river_profile = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd = read_positive_ead_usd(RIVER_EAD_MAX_PATH, river_profile)[0]
river_transform = river_profile["transform"]
river_shape = (river_profile["height"], river_profile["width"])
river_pixel_area_ha = abs(river_transform.a * river_transform.e) / 10_000.0
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)

ndvi_before = reproject_continuous_to_reference(NDVI_BEFORE_PATH, river_profile)
ndvi_after = reproject_continuous_to_reference(NDVI_AFTER_PATH, river_profile)
forest_change_gt10_class = reproject_categorical_to_reference(FOREST_CHANGE_GT10_PATH, river_profile)

landcover = gpd.read_file(LANDCOVER_PATH, columns=["Classify", "geometry"]).to_crs(river_profile["crs"])
landcover = landcover[landcover.geometry.notnull() & ~landcover.geometry.is_empty].copy()
landcover["Classify"] = landcover["Classify"].astype(str)
landcover["forest_fraction"] = landcover["Classify"].map(
    lambda class_name: forest_fraction_for_class(
        class_name,
        forest_flood_equivalent_classes,
        mixed_land_use_fractions_primary,
    )
)
landcover["class_id"] = pd.factorize(landcover["Classify"], sort=True)[0] + 1
class_lookup = landcover[["class_id", "Classify", "forest_fraction"]].drop_duplicates().sort_values("class_id")
class_id_to_name = dict(zip(class_lookup["class_id"], class_lookup["Classify"]))

class_shapes = ((geometry, int(class_id)) for geometry, class_id in zip(landcover.geometry, landcover["class_id"]))
landcover_class_grid = rasterize(
    shapes=class_shapes,
    out_shape=river_shape,
    transform=river_transform,
    fill=0,
    dtype="int16",
)

forest_fraction_shapes = ((geometry, float(forest_fraction)) for geometry, forest_fraction in zip(landcover.geometry, landcover["forest_fraction"]))
forest_fraction_grid = rasterize(
    shapes=forest_fraction_shapes,
    out_shape=river_shape,
    transform=river_transform,
    fill=0.0,
    dtype="float32",
)

pd.DataFrame(
    [
        {
            "river_benefit_area_ha": river_benefit_mask.sum() * river_pixel_area_ha,
            "river_grid_pixel_area_ha": river_pixel_area_ha,
            "landcover_pixels_on_grid": int((landcover_class_grid > 0).sum()),
        }
    ]
)


In [ ]:
ndvi_eligible_mask = (
    river_benefit_mask
    & np.isfinite(ndvi_before)
    & np.isfinite(ndvi_after)
    & (ndvi_before >= REL_BASELINE_MIN)
)
relative_ndvi_change = np.full(river_shape, np.nan, dtype="float32")
np.divide(
    ndvi_after - ndvi_before,
    ndvi_before,
    out=relative_ndvi_change,
    where=ndvi_eligible_mask,
)

river_greening_mask = ndvi_eligible_mask & (relative_ndvi_change >= REL_GREENING_THRESHOLD)
forest_gt10_increase_mask = forest_change_gt10_class == 1
strict_forest_gt10_river_benefit_mask = river_benefit_mask & forest_gt10_increase_mask

river_greening_area_ha = river_greening_mask.sum() * river_pixel_area_ha
strict_forest_gt10_river_benefit_area_ha = strict_forest_gt10_river_benefit_mask.sum() * river_pixel_area_ha
river_greening_ead_min = ead_sum_usd(river_ead_min_usd, river_greening_mask)
river_greening_ead_max = ead_sum_usd(river_ead_max_usd, river_greening_mask)

mask_summary = pd.DataFrame(
    [
        {
            "mask": "River-flood restoration-benefit pixels with NDVI increase >=10%",
            "area_ha": river_greening_area_ha,
            "avoided_ead_usd_minimum": river_greening_ead_min,
            "avoided_ead_usd_maximum": river_greening_ead_max,
        },
        {
            "mask": "Strict forest-equivalent >10% greening overlapped with river-benefit pixels",
            "area_ha": strict_forest_gt10_river_benefit_area_ha,
            "avoided_ead_usd_minimum": ead_sum_usd(river_ead_min_usd, strict_forest_gt10_river_benefit_mask),
            "avoided_ead_usd_maximum": ead_sum_usd(river_ead_max_usd, strict_forest_gt10_river_benefit_mask),
        },
    ]
)
mask_summary


In [ ]:
def summarise_mask_by_landcover(mask: np.ndarray, scope: str, use_forest_fraction_weight: bool) -> pd.DataFrame:
    rows = []
    for class_id, class_name in class_id_to_name.items():
        class_mask = mask & (landcover_class_grid == class_id)
        if use_forest_fraction_weight:
            area_ha = float(forest_fraction_grid[class_mask].sum() * river_pixel_area_ha)
        else:
            area_ha = float(class_mask.sum() * river_pixel_area_ha)
        if area_ha <= 0:
            continue

        rows.append(
            {
                "scope": scope,
                "Classify": class_name,
                "landcover_group": landcover_group_for_class(class_name),
                "forest_fraction_used": float(class_lookup.loc[class_lookup["class_id"].eq(class_id), "forest_fraction"].iloc[0]),
                "area_ha": area_ha,
                "avoided_ead_usd_minimum": ead_sum_usd(river_ead_min_usd, class_mask),
                "avoided_ead_usd_maximum": ead_sum_usd(river_ead_max_usd, class_mask),
            }
        )

    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary

    total_area_ha = summary["area_ha"].sum()
    total_ead_min = summary["avoided_ead_usd_minimum"].sum()
    total_ead_max = summary["avoided_ead_usd_maximum"].sum()
    summary["pct_of_scope_area"] = summary["area_ha"].map(lambda value: pct(value, total_area_ha))
    summary["pct_of_scope_ead_minimum"] = summary["avoided_ead_usd_minimum"].map(lambda value: pct(value, total_ead_min))
    summary["pct_of_scope_ead_maximum"] = summary["avoided_ead_usd_maximum"].map(lambda value: pct(value, total_ead_max))
    return summary.sort_values("area_ha", ascending=False).reset_index(drop=True)


river_benefit_greening_by_landcover = summarise_mask_by_landcover(
    river_greening_mask,
    "River-flood restoration-benefit pixels with NDVI increase >=10%",
    use_forest_fraction_weight=False,
)
river_benefit_greening_forest_weighted_by_landcover = summarise_mask_by_landcover(
    river_greening_mask,
    "River-flood restoration-benefit pixels with NDVI increase >=10%, forest-fraction weighted",
    use_forest_fraction_weight=True,
)
strict_forest_overlap_by_landcover = summarise_mask_by_landcover(
    strict_forest_gt10_river_benefit_mask,
    "Strict forest-equivalent >10% greening overlapped with river-benefit pixels",
    use_forest_fraction_weight=False,
)

print("River-benefit greening by 2013 land-cover class")
display(river_benefit_greening_by_landcover.round(3))
print("Strict forest-equivalent greening overlap by 2013 land-cover class")
display(strict_forest_overlap_by_landcover.round(3))


In [ ]:
forest_increase_by_class_weighted = pd.read_csv(FOREST_INCREASE_BY_CLASS_WEIGHTED_PATH)
forest_increase_by_class_weighted["landcover_group"] = forest_increase_by_class_weighted["Classify"].map(landcover_group_for_class)
forest_increase_gt10_total_area_ha = forest_increase_by_class_weighted["substantial_increase_gt10pct_area_ha"].sum()
forest_increase_by_class_weighted["pct_of_substantial_increase_area"] = forest_increase_by_class_weighted[
    "substantial_increase_gt10pct_area_ha"
].map(lambda value: pct(value, forest_increase_gt10_total_area_ha))
forest_increase_by_class_weighted = forest_increase_by_class_weighted.sort_values(
    "substantial_increase_gt10pct_area_ha",
    ascending=False,
).reset_index(drop=True)

wider_forest_greening_group_summary = (
    forest_increase_by_class_weighted.groupby("landcover_group", as_index=False)["substantial_increase_gt10pct_area_ha"].sum()
    .sort_values("substantial_increase_gt10pct_area_ha", ascending=False)
    .reset_index(drop=True)
)
wider_forest_greening_group_summary["pct_of_scope_area"] = wider_forest_greening_group_summary[
    "substantial_increase_gt10pct_area_ha"
].map(lambda value: pct(value, forest_increase_gt10_total_area_ha))

river_benefit_greening_group_summary = (
    river_benefit_greening_by_landcover.groupby("landcover_group", as_index=False)["area_ha"].sum()
    .sort_values("area_ha", ascending=False)
    .reset_index(drop=True)
)
river_benefit_greening_group_summary["pct_of_scope_area"] = river_benefit_greening_group_summary["area_ha"].map(
    lambda value: pct(value, river_benefit_greening_group_summary["area_ha"].sum())
)

strict_forest_overlap_group_summary = (
    strict_forest_overlap_by_landcover.groupby("landcover_group", as_index=False)["area_ha"].sum()
    .sort_values("area_ha", ascending=False)
    .reset_index(drop=True)
)
strict_forest_overlap_group_summary["pct_of_scope_area"] = strict_forest_overlap_group_summary["area_ha"].map(
    lambda value: pct(value, strict_forest_overlap_group_summary["area_ha"].sum())
)

print("All-Jamaica weighted forest-equivalent >10% greening by grouped land cover")
display(wider_forest_greening_group_summary.round(3))
print("River-benefit greening by grouped land cover")
display(river_benefit_greening_group_summary.round(3))
print("Strict forest-equivalent greening and river-benefit overlap by grouped land cover")
display(strict_forest_overlap_group_summary.round(3))


In [ ]:
group_plot_data = pd.concat(
    [
        wider_forest_greening_group_summary.rename(
            columns={"substantial_increase_gt10pct_area_ha": "plot_area_ha"}
        ).assign(scope="All forest-equivalent >10% greening")[["scope", "landcover_group", "plot_area_ha", "pct_of_scope_area"]],
        river_benefit_greening_group_summary.assign(scope="River-benefit >10% greening")[[
            "scope",
            "landcover_group",
            "area_ha",
            "pct_of_scope_area",
        ]].rename(columns={"area_ha": "plot_area_ha"}),
        strict_forest_overlap_group_summary.assign(scope="Strict forest-equivalent overlap")[[
            "scope",
            "landcover_group",
            "area_ha",
            "pct_of_scope_area",
        ]].rename(columns={"area_ha": "plot_area_ha"}),
    ],
    ignore_index=True,
)

scope_order = [
    "All forest-equivalent >10% greening",
    "River-benefit >10% greening",
    "Strict forest-equivalent overlap",
]
group_colors = {
    "Open dry forest": "#9b6a2f",
    "Plantation / tree crops": "#2b8c57",
    "Secondary / broadleaved forest": "#66a61e",
    "Mixed fields, bamboo and secondary forest": "#d8a934",
    "Open / agricultural fields": "#bdbdbd",
    "Bauxite extraction / quarry": "#c66b35",
    "Bamboo": "#7fcdbb",
}

fig, axes = plt.subplots(len(scope_order), 1, figsize=(180 / 25.4, 118 / 25.4), dpi=FIGURE_DPI)
for axis, scope in zip(axes, scope_order, strict=True):
    scope_data = group_plot_data[group_plot_data["scope"].eq(scope)].sort_values("plot_area_ha", ascending=True)
    axis.barh(
        scope_data["landcover_group"],
        scope_data["plot_area_ha"],
        color=scope_data["landcover_group"].map(group_colors),
        edgecolor="white",
        linewidth=0.3,
    )
    for row_index, row in enumerate(scope_data.itertuples(index=False)):
        axis.text(
            row.plot_area_ha,
            row_index,
            f" {row.pct_of_scope_area:.1f}%",
            va="center",
            ha="left",
            fontsize=6,
        )
    axis.set_title(scope, pad=3)
    axis.set_xlabel("Area (ha)")
    axis.grid(axis="x", linewidth=0.3, alpha=0.35)
    axis.tick_params(axis="y", labelsize=6)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.set_xlim(0, scope_data["plot_area_ha"].max() * 1.20)

fig.suptitle("2013 land cover associated with post-Hurricane Melissa NDVI greening", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.955], h_pad=1.0)

figure_paths = []
for suffix in ["png", "pdf", "svg"]:
    figure_path = OUT_DIR / f"hurricane_melissa_forest_greening_landcover_summary.{suffix}"
    fig.savefig(figure_path, bbox_inches="tight", facecolor="white")
    figure_paths.append(figure_path)

display(fig)
plt.close(fig)
figure_paths


In [ ]:
output_paths = []
output_tables = {
    "river_benefit_greening_by_landcover.csv": river_benefit_greening_by_landcover,
    "river_benefit_greening_forest_weighted_by_landcover.csv": river_benefit_greening_forest_weighted_by_landcover,
    "strict_forest_gt10_greening_river_benefit_by_landcover.csv": strict_forest_overlap_by_landcover,
    "all_jamaica_forest_gt10_greening_by_landcover_weighted.csv": forest_increase_by_class_weighted,
    "wider_forest_greening_group_summary.csv": wider_forest_greening_group_summary,
    "river_benefit_greening_group_summary.csv": river_benefit_greening_group_summary,
    "strict_forest_overlap_group_summary.csv": strict_forest_overlap_group_summary,
    "mask_summary.csv": mask_summary,
}
for filename, table in output_tables.items():
    output_path = OUT_DIR / filename
    table.to_csv(output_path, index=False)
    output_paths.append(output_path)

output_paths


In [ ]:
dry_forest_area_wider_ha = wider_forest_greening_group_summary.loc[
    wider_forest_greening_group_summary["landcover_group"].eq("Open dry forest"),
    "substantial_increase_gt10pct_area_ha",
].sum()
dry_forest_area_river_ha = river_benefit_greening_group_summary.loc[
    river_benefit_greening_group_summary["landcover_group"].eq("Open dry forest"),
    "area_ha",
].sum()
open_agriculture_area_river_ha = river_benefit_greening_group_summary.loc[
    river_benefit_greening_group_summary["landcover_group"].eq("Open / agricultural fields"),
    "area_ha",
].sum()
mixed_area_river_ha = river_benefit_greening_group_summary.loc[
    river_benefit_greening_group_summary["landcover_group"].eq("Mixed fields, bamboo and secondary forest"),
    "area_ha",
].sum()
plantation_area_wider_ha = wider_forest_greening_group_summary.loc[
    wider_forest_greening_group_summary["landcover_group"].eq("Plantation / tree crops"),
    "substantial_increase_gt10pct_area_ha",
].sum()
plantation_pct_wider = wider_forest_greening_group_summary.loc[
    wider_forest_greening_group_summary["landcover_group"].eq("Plantation / tree crops"),
    "pct_of_scope_area",
].sum()

summary_text = f"""The greening signal differs depending on the mask being interpreted. Across the wider all-Jamaica forest-equivalent >10% greening signal, open dry forest accounted for {dry_forest_area_wider_ha:,.1f} ha ({pct(dry_forest_area_wider_ha, forest_increase_gt10_total_area_ha):.1f}%), almost all of which was tall open dry forest. Plantation/tree-crop classes accounted for {plantation_area_wider_ha:,.1f} ha ({plantation_pct_wider:.1f}%).

For the river-flood restoration-benefit pixels used in the results paragraph, however, the >10% greening area was not mapped as dry forest in the 2013 land-cover layer. Open dry forest accounted for {dry_forest_area_river_ha:,.1f} ha ({pct(dry_forest_area_river_ha, river_greening_area_ha):.1f}%). Instead, the greening pixels were mainly open/agricultural fields ({open_agriculture_area_river_ha:,.1f} ha; {pct(open_agriculture_area_river_ha, river_greening_area_ha):.1f}%) and mixed fields, bamboo and secondary forest ({mixed_area_river_ha:,.1f} ha; {pct(mixed_area_river_ha, river_greening_area_ha):.1f}%). The largest single class was herbaceous crops/fallow/cultivated vegetables.
"""

summary_path = OUT_DIR / "hurricane_melissa_forest_greening_landcover_interpretation.md"
summary_path.write_text(summary_text, encoding="utf-8")
print(summary_text)
print(f"Wrote: {summary_path}")
